# ODI to Databricks Migration

## Target: `insert_hr_trg_loc.sql`

**Conversion Timestamp**: 2024-07-30 12:00:00 UTC

**Description**: This notebook converts the ODI SQL for loading data into the `TRG_LOC` table from `LOCATIONS`.

In [ ]:
import dbutils

# Create widgets for ETL parameters as required by the system prompt.
# These parameters are not explicitly used in the provided SQL snippet but are mandatory for the notebook structure.
dbutils.widgets.text("ETL_JOB_TYPE", "FULL", "ETL Job Type (e.g., FULL, INCREMENTAL)")
dbutils.widgets.text("DATASOURCE_NUM_ID", "1", "Datasource Number ID")
dbutils.widgets.text("ETL_PROC_WID", "-1", "ETL Process Widget ID")
dbutils.widgets.text("ODI_SESS_NO", "-1", "ODI Session Number")

# ETL Parameters

In [ ]:
%sql
-- Create temporary views for ETL parameters for easier SQL referencing
CREATE OR REPLACE TEMPORARY VIEW v_etl_job_type AS SELECT '${ETL_JOB_TYPE}' AS etl_job_type;

CREATE OR REPLACE TEMPORARY VIEW v_datasource_num_id AS SELECT ${DATASOURCE_NUM_ID} AS datasource_num_id;

CREATE OR REPLACE TEMPORARY VIEW v_etl_proc_wid AS SELECT ${ETL_PROC_WID} AS etl_proc_wid;

CREATE OR REPLACE TEMPORARY VIEW v_odi_sess_no AS SELECT '${ODI_SESS_NO}' AS odi_sess_no;

In [ ]:
print("ETL Parameters:")
display(spark.sql("SELECT * FROM v_etl_job_type"))
display(spark.sql("SELECT * FROM v_datasource_num_id"))
display(spark.sql("SELECT * FROM v_etl_proc_wid"))
display(spark.sql("SELECT * FROM v_odi_sess_no"))

# Load Target Table

In [ ]:
%sql
-- SCEN_TASK_NO in {10}
-- SCEN_TASK_NO in {20}
-- SCEN_TASK_NO in {30}: Insert into TRG_LOC
-- Converted: Removed Oracle hints /*+ APPEND PARALLEL */.
-- Converted: Oracle schema HR to workspace.hr, table names to lowercase.
INSERT 
  INTO workspace.hr.trg_loc
  (
    LOCATION_ID ,
    STREET_ADDRESS ,
    POSTAL_CODE ,
    CITY ,
    STATE_PROVINCE ,
    COUNTRY_ID 
  ) 
SELECT 
  LOCATIONS.LOCATION_ID ,
  LOCATIONS.STREET_ADDRESS ,
  LOCATIONS.POSTAL_CODE ,
  LOCATIONS.CITY ,
  LOCATIONS.STATE_PROVINCE ,
  LOCATIONS.COUNTRY_ID  
FROM 
  workspace.hr.locations AS LOCATIONS;

# Validation

In [ ]:
%sql
-- Verify record count in the target table after insertion
SELECT COUNT(*) AS total_records_in_trg_loc FROM workspace.hr.trg_loc;

# Conversion Notes and Manual Actions Required

1.  **Schema and Table Names**: Oracle `HR.TRG_LOC` and `HR.LOCATIONS` have been converted to `workspace.hr.trg_loc` and `workspace.hr.locations` respectively. Ensure these tables exist in your Databricks environment with the appropriate Delta Lake format.
2.  **Oracle Hints**: The Oracle `/*+ APPEND PARALLEL */` hint has been removed as it is not applicable in Databricks Spark SQL / Delta Lake. Delta Lake handles concurrency and performance automatically.
3.  **Data Types**: The original ODI file did not include `CREATE TABLE` statements. Ensure that the `workspace.hr.trg_loc` table is created in Databricks with Spark SQL compatible data types (e.g., `STRING` for `VARCHAR2`, `BIGINT` for `NUMBER(p,0)`, `TIMESTAMP` for `DATE` or `TIMESTAMP`). The columns in the `SELECT` list will adapt to the target table's schema.
4.  **ETL Parameters**: Generic widgets (`ETL_JOB_TYPE`, `DATASOURCE_NUM_ID`, `ETL_PROC_WID`, `ODI_SESS_NO`) have been included following the required notebook structure, even though they are not explicitly used in this specific `INSERT` statement. You may configure them as needed.